# Step 4: Topic Modeling
## YouTube Fast Fashion Intelligence Engine — CSCI370

### What is Topic Modeling?

Topic modeling is an **unsupervised** NLP technique — meaning you do not tell the model
what topics to look for. It figures them out automatically by finding groups of comments
that use similar words.

For example, given thousands of comments it might discover on its own:
- A cluster of comments talking about *"workers, wages, factories, Bangladesh"* → **Labor Conditions**
- A cluster talking about *"plastic, packaging, waste, ocean"* → **Environmental Impact**
- A cluster talking about *"haul, try-on, review, cheap"* → **Shopping Culture**

### What tool we use: BERTopic

BERTopic works in 3 stages:
1. **Embed** — convert each comment into a vector (list of numbers representing meaning)
2. **Cluster** — group similar vectors together using UMAP + HDBSCAN algorithms
3. **Label** — find the most representative words for each cluster using TF-IDF

### What we do in this step:
1. General topic modeling on **all 30,609 comments**
2. Per-sentiment topic modeling — run BERTopic **separately** on Positive, Negative, and Neutral comments
3. Interpret and give meaningful names to each discovered topic

### Note on the embedding model
Ideally we would use `sentence-transformers/all-MiniLM-L6-v2` from HuggingFace for embeddings.
If you have internet access, uncomment the HuggingFace version in the setup cell.
We also provide an offline fallback using TF-IDF + SVD which works without any downloads.


## 0. Install Libraries

In [1]:
!pip install bertopic scikit-learn umap-learn hdbscan -q


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load Libraries and Data

In [2]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from bertopic.backend import BaseEmbedder
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Load the English-only dataset (output of Step 2b)
df = pd.read_csv('youtube_comments_english.csv')
df = df.dropna(subset=['text_clean']).reset_index(drop=True)

print(f"Loaded {len(df):,} comments")
print(f"Sentiment breakdown:")
print(df['sentiment_label'].value_counts())
print(f"\nColumns: {df.columns.tolist()}")

C:\Users\AK Khan\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 30,609 comments
Sentiment breakdown:
sentiment_label
Positive    14554
Negative    11958
Neutral      4097
Name: count, dtype: int64

Columns: ['author', 'updated_at', 'like_count', 'text', 'video_id', 'public', 'text_clean', 'sentiment_score', 'sentiment_label', 'sentiment_source', 'language']


## 2. Set Up the Embedding Model

We provide two options:

**Option A (recommended if you have internet):** Use `sentence-transformers` from HuggingFace.
This gives better quality topics because it understands meaning, not just word frequency.

**Option B (offline fallback):** Use TF-IDF + SVD.
Works without any downloads. Topics are still meaningful but slightly less nuanced.

Uncomment whichever option applies to you.


In [3]:
# ── OPTION A: HuggingFace sentence-transformers (better quality) ─────────────
# Uncomment these lines if you have internet access

# from sentence_transformers import SentenceTransformer
# embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
# print("Using HuggingFace sentence-transformers")


# ── OPTION B: Offline TF-IDF + SVD fallback ──────────────────────────────────
# This runs without any downloads — use if HuggingFace is unavailable

class TFIDFEmbedder(BaseEmbedder):
    """
    Custom embedding model using TF-IDF + SVD.
    TF-IDF converts text to numbers based on word frequency.
    SVD reduces those numbers to 50 dimensions (faster clustering).
    No internet required.
    """
    def __init__(self, n_components=50):
        self.tfidf = TfidfVectorizer(
            max_features=8000,
            stop_words='english',
            ngram_range=(1, 2),   # use single words and 2-word phrases
            min_df=3              # ignore words appearing in fewer than 3 comments
        )
        self.svd = TruncatedSVD(n_components=n_components, random_state=42)

    def embed(self, documents, verbose=False):
        X = self.tfidf.fit_transform(documents)
        return self.svd.fit_transform(X)

embedding_model = TFIDFEmbedder()
print("Using offline TF-IDF + SVD embedder")

Using offline TF-IDF + SVD embedder


## 3. Configure BERTopic

We set up BERTopic with these key parameters:

- `nr_topics` — how many topics to find. We set 15 for the full dataset.
  BERTopic may find fewer if the data doesn't support that many.
- `min_topic_size` — minimum number of comments to form a topic (avoids tiny meaningless clusters)
- `vectorizer_model` — how to represent topics as words (we use bigrams so we get phrases like "fast fashion" not just "fast" and "fashion" separately)
- Topic `-1` is the **outlier topic** — comments BERTopic couldn't fit into any cluster. This is normal and expected.


In [4]:
def build_topic_model(nr_topics=15, min_topic_size=20):
    """
    Build and return a configured BERTopic model.
    We use a function so we can easily create fresh models for each sentiment group.
    """
    vectorizer = CountVectorizer(
        stop_words='english',
        ngram_range=(1, 2),   # single words + 2-word phrases
        min_df=3
    )

    model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer,
        nr_topics=nr_topics,
        min_topic_size=min_topic_size,
        verbose=False
    )
    return model

print("BERTopic model builder ready")

BERTopic model builder ready


## 4. General Topic Modeling — All Comments

We run BERTopic on all 30,609 comments to find the overall themes
people discuss when talking about fast fashion on YouTube.


In [6]:
docs_all = df['text_clean'].tolist()

print(f"Running BERTopic on {len(docs_all):,} comments...")
model_all = build_topic_model(nr_topics=15, min_topic_size=30)
topics_all, probs_all = model_all.fit_transform(docs_all)

# Add topic assignments back to the dataframe
df['topic_general'] = topics_all

print("Done!")
print(f"\nNumber of topics found: {model_all.get_topic_info().shape[0] - 1} (excluding outliers)")
print(f"Outlier comments (topic -1): {(np.array(topics_all) == -1).sum():,}")

Running BERTopic on 30,609 comments...
Done!

Number of topics found: 14 (excluding outliers)
Outlier comments (topic -1): 7,922


### 4a. View the Discovered Topics

Each topic is represented by its top keywords — these are the words that
appear most distinctively in that cluster of comments.


In [7]:
topic_info = model_all.get_topic_info()

print("All discovered topics:")
print("-" * 65)
for _, row in topic_info.iterrows():
    if row['Topic'] == -1:
        print(f"  Topic -1 (outliers): {row['Count']:,} comments — no clear theme")
        continue
    words = model_all.get_topic(row['Topic'])
    top_words = [w for w, s in words[:7]]
    print(f"  Topic {row['Topic']:2d} ({row['Count']:4d} comments): {', '.join(top_words)}")

All discovered topics:
-----------------------------------------------------------------
  Topic -1 (outliers): 7,922 comments — no clear theme
  Topic  0 (11437 comments): fashion, fast, fast fashion, clothes, people, like, just
  Topic  1 (4604 comments): hunter, know, children, sad, american, shirt, lol
  Topic  2 (1806 comments): video, great, really, love, good, great video, like
  Topic  3 (1638 comments): shein, buy, got, yes, ad, india, temu
  Topic  4 ( 659 comments): hot, hot topic, topic, manager, store, like, employees
  Topic  5 ( 515 comments): thank, video, sharing, thank sharing, video thank, thank video, thank making
  Topic  6 ( 365 comments): heard, heard shein, ve, ve heard, shein, shop, shopped
  Topic  7 ( 340 comments): right, married, realize, just, like, comment, don
  Topic  8 ( 282 comments): meat, papa, papa meat, oh, merch, ranking, meat canyon
  Topic  9 ( 279 comments): unicef, patel, wu, ey, hii, proverty, provincelife8
  Topic 10 ( 225 comments): thanks

### 4b. Give Meaningful Names to Each Topic

BERTopic gives topics numbers (0, 1, 2...) and keywords.
We look at those keywords and assign a human-readable label.

**This is the "interpret and label" requirement from your project.**

Look at the keywords printed above, then fill in the `topic_labels` dictionary below.
We've pre-filled some examples — update them based on what YOUR model actually found.


In [8]:
# ── UPDATE THESE LABELS based on the keywords printed above ─────────────────
# Format: topic_number: 'Your Label'
# Look at the keywords for each topic and give it a meaningful name

topic_labels = {
    -1: 'Outliers / No Clear Theme',
    0:  'Fast Fashion Shopping Habits',      # e.g. clothes, buy, fashion, cheap
    1:  'Labor & Worker Conditions',         # e.g. workers, wages, factories, china
    2:  'Environmental Impact',              # e.g. waste, plastic, environment, ocean
    3:  'Shein Brand Discussion',            # e.g. shein, quality, shipping, haul
    4:  'Temu Brand Discussion',             # e.g. temu, prices, app, order
    5:  'Consumer Guilt & Ethics',           # e.g. guilty, conscious, ethical, aware
    6:  'Video Reactions',                   # e.g. video, great, thanks, watched
    7:  'Price & Affordability',             # e.g. cheap, afford, price, expensive
    8:  'Social Media & Influencers',        # e.g. influencer, haul, youtube, tiktok
    9:  'Sustainability & Alternatives',     # e.g. thrift, sustainable, secondhand
    10: 'Political & Trade Commentary',      # e.g. tariffs, trade, government, america
    # Add more as needed based on your output
}

# Apply labels to dataframe
df['topic_general_label'] = df['topic_general'].map(
    lambda t: topic_labels.get(t, f'Topic {t}')
)

print("Topic label mapping:")
for tid, label in topic_labels.items():
    count = (df['topic_general'] == tid).sum()
    if count > 0:
        print(f"  Topic {tid:2d} ({count:4d} comments) → {label}")

Topic label mapping:
  Topic -1 (7922 comments) → Outliers / No Clear Theme
  Topic  0 (11437 comments) → Fast Fashion Shopping Habits
  Topic  1 (4604 comments) → Labor & Worker Conditions
  Topic  2 (1806 comments) → Environmental Impact
  Topic  3 (1638 comments) → Shein Brand Discussion
  Topic  4 ( 659 comments) → Temu Brand Discussion
  Topic  5 ( 515 comments) → Consumer Guilt & Ethics
  Topic  6 ( 365 comments) → Video Reactions
  Topic  7 ( 340 comments) → Price & Affordability
  Topic  8 ( 282 comments) → Social Media & Influencers
  Topic  9 ( 279 comments) → Sustainability & Alternatives
  Topic 10 ( 225 comments) → Political & Trade Commentary


In [9]:
# Show 2 real example comments from each topic so you can verify labels make sense
print("Sample comments per topic (for label verification):")
print("=" * 70)

for topic_id in sorted(df['topic_general'].unique()):
    if topic_id == -1:
        continue
    label = topic_labels.get(topic_id, f'Topic {topic_id}')
    samples = df[df['topic_general'] == topic_id]['text_clean'].sample(
        min(2, (df['topic_general'] == topic_id).sum()), random_state=42
    ).tolist()
    print(f"\n  [{label}]")
    for s in samples:
        print(f"  → {s[:150]}")

Sample comments per topic (for label verification):

  [Fast Fashion Shopping Habits]
  → Outlets are different than the actual stores though
  → @gressos Being Portuguese doesn’t make anyone automatically correct about history especially colonial history where men in that time are now dead, tha

  [Labor & Worker Conditions]
  → first time
  → And where women and children are raped by chuslamist continuously!!!

  [Environmental Impact]
  → I do LOVE the title card sequences and how they fit into the set - kudos for imagination and planning that out! Now, I'd love to leave a thoughtful co
  → I love that Teen Vogue is putting out content that allows its audience to become more aware of reality. I am HERE for this kind of content!!!!

  [Shein Brand Discussion]
  → Never shopped with Shein and never will.
  → Don't buy clothes by Shein, Temu etc. Pleas!!!! Respect for this childern!!!!! 😢😢😢

  [Temu Brand Discussion]
  → @AndresMoreno-b4rshawty, are you really going to get into a legal

## 5. Per-Sentiment Topic Modeling

Now we split the dataset by sentiment label and run BERTopic separately on each group.

This answers the question: **do people talk about different things depending on how they feel?**

- Negative comments → what upsets people about fast fashion?
- Positive comments → what do supporters appreciate?
- Neutral comments → what are people just observing or discussing factually?


In [10]:
sentiment_groups = {
    'Positive': df[df['sentiment_label'] == 'Positive']['text_clean'].tolist(),
    'Negative': df[df['sentiment_label'] == 'Negative']['text_clean'].tolist(),
    'Neutral':  df[df['sentiment_label'] == 'Neutral']['text_clean'].tolist(),
}

print("Comment counts per sentiment group:")
for label, docs in sentiment_groups.items():
    print(f"  {label}: {len(docs):,} comments")

Comment counts per sentiment group:
  Positive: 14,554 comments
  Negative: 11,958 comments
  Neutral: 4,097 comments


In [11]:
# Run BERTopic separately on each sentiment group
# We store the model and topic assignments for each group

sentiment_models  = {}
sentiment_topics  = {}

for sentiment, docs in sentiment_groups.items():
    print(f"\nRunning BERTopic on {sentiment} comments ({len(docs):,})...")

    # Use fewer topics for smaller groups
    nr = 10 if len(docs) < 8000 else 12
    min_size = 15 if len(docs) < 8000 else 20

    model = build_topic_model(nr_topics=nr, min_topic_size=min_size)
    topics, probs = model.fit_transform(docs)

    sentiment_models[sentiment] = model
    sentiment_topics[sentiment] = topics

    n_topics = model.get_topic_info().shape[0] - 1
    n_outliers = sum(1 for t in topics if t == -1)
    print(f"  Found {n_topics} topics, {n_outliers:,} outlier comments")

print("\nAll sentiment groups done!")


Running BERTopic on Positive comments (14,554)...
  Found 11 topics, 3,330 outlier comments

Running BERTopic on Negative comments (11,958)...
  Found 11 topics, 2,389 outlier comments

Running BERTopic on Neutral comments (4,097)...
  Found 9 topics, 322 outlier comments

All sentiment groups done!


In [12]:
# Print topics for each sentiment group
for sentiment, model in sentiment_models.items():
    print(f"\n{'='*65}")
    print(f"  {sentiment.upper()} COMMENTS — Topics")
    print(f"{'='*65}")

    topic_info = model.get_topic_info()
    for _, row in topic_info.iterrows():
        if row['Topic'] == -1:
            continue
        words = model.get_topic(row['Topic'])
        top_words = [w for w, s in words[:7]]
        print(f"  Topic {row['Topic']:2d} ({row['Count']:4d} comments): {', '.join(top_words)}")


  POSITIVE COMMENTS — Topics
  Topic  0 (5969 comments): fashion, clothes, fast, fast fashion, like, people, buy
  Topic  1 (1886 comments): hunter, hot, hot topic, topic, shirt, like, looks
  Topic  2 (1398 comments): video, great, thanks, love, good, really, great video
  Topic  3 ( 603 comments): thank, journalism, sharing, video, thank sharing, video thank, thank video
  Topic  4 ( 458 comments): god, help, bless, jesus, god bless, children, people
  Topic  5 ( 261 comments): wow, true, documentary, true cost, good true, video, cost
  Topic  6 ( 201 comments): yes, shein, cheap, heard, buy, women, heard shein
  Topic  7 ( 155 comments): agree, completely agree, totally agree, totally, completely, just, think
  Topic  8 ( 147 comments): meat, papa, papa meat, merch, video, rank, ranking
  Topic  9 (  80 comments): exactly, design, production, know exactly, highest, buyers, yessss
  Topic 10 (  66 comments): geography, arm, pumpkin, coolest monkey, columbus ohio, bitches, monkey

  

### 5a. Label the Per-Sentiment Topics

Same as before — look at the keywords above and give each topic a meaningful name.
Update the dictionaries below based on what YOUR model found.


In [13]:
# ── UPDATE THESE based on the keywords printed above ─────────────────────────

sentiment_topic_labels = {
    'Positive': {
        -1: 'Outliers',
        0:  'Affordable Fashion Finds',
        1:  'Brand Appreciation',
        2:  'Positive Shopping Experience',
        3:  'Influencer Praise',
        # Add more based on your output
    },
    'Negative': {
        -1: 'Outliers',
        0:  'Worker Exploitation',
        1:  'Environmental Destruction',
        2:  'Poor Quality Products',
        3:  'Ethical Concerns',
        4:  'Anti-Fast Fashion Sentiment',
        # Add more based on your output
    },
    'Neutral': {
        -1: 'Outliers',
        0:  'Brand Comparisons',
        1:  'Industry Observations',
        2:  'Shipping & Logistics',
        3:  'Price Discussions',
        # Add more based on your output
    }
}

print("Per-sentiment labels set.")
print("Remember to update these dictionaries based on the keywords shown above!")

Per-sentiment labels set.
Remember to update these dictionaries based on the keywords shown above!


## 6. Key Insight — Compare Topics Across Sentiments

This is the most valuable output for your dashboard and report.
We compare what each sentiment group talks about.


In [14]:
print("CROSS-SENTIMENT TOPIC COMPARISON")
print("=" * 65)
print("What are the TOP keywords for each sentiment group?")
print()

for sentiment, model in sentiment_models.items():
    topic_info = model.get_topic_info()
    # Get top 3 non-outlier topics by size
    top_topics = topic_info[topic_info['Topic'] != -1].head(3)
    labels = sentiment_topic_labels.get(sentiment, {})

    print(f"{sentiment.upper()} — top themes:")
    for _, row in top_topics.iterrows():
        words = model.get_topic(row['Topic'])
        top_words = [w for w, s in words[:5]]
        label = labels.get(row['Topic'], f"Topic {row['Topic']}")
        print(f"  [{label}] → {', '.join(top_words)}")
    print()

CROSS-SENTIMENT TOPIC COMPARISON
What are the TOP keywords for each sentiment group?

POSITIVE — top themes:
  [Affordable Fashion Finds] → fashion, clothes, fast, fast fashion, like
  [Brand Appreciation] → hunter, hot, hot topic, topic, shirt
  [Positive Shopping Experience] → video, great, thanks, love, good

NEGATIVE — top themes:
  [Worker Exploitation] → people, fashion, fast, fast fashion, clothes
  [Environmental Destruction] → video, shit, way, really, hunter
  [Poor Quality Products] → world, slavery, sad, horrible, modern

NEUTRAL — top themes:
  [Brand Comparisons] → clothes, fashion, buy, fast, fast fashion
  [Industry Observations] → shein, video, heard, temu, zara
  [Shipping & Logistics] → shirt, alaska, live, 16, law



## 7. Save Results

In [15]:
# Add per-sentiment topic columns to the dataframe
# We need to map back using the index of each sentiment group

for sentiment, docs in sentiment_groups.items():
    # Get the indices of comments belonging to this sentiment group
    indices = df[df['sentiment_label'] == sentiment].index.tolist()
    topics  = sentiment_topics[sentiment]

    col_name = f'topic_{sentiment.lower()}'
    # Initialize column with -1
    if col_name not in df.columns:
        df[col_name] = -1

    for idx, topic in zip(indices, topics):
        df.at[idx, col_name] = topic

# Save
output_path = 'youtube_comments_topics.csv'
df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Total rows: {len(df):,}")
print(f"\nNew columns added:")
new_cols = ['topic_general', 'topic_general_label', 'topic_positive', 'topic_negative', 'topic_neutral']
for col in new_cols:
    if col in df.columns:
        print(f"  {col}")
print("\nPreview:")
df[['text_clean', 'sentiment_label', 'topic_general_label']].head(5)

Saved to: youtube_comments_topics.csv
Total rows: 30,609

New columns added:
  topic_general
  topic_general_label
  topic_positive
  topic_negative
  topic_neutral

Preview:


,text_clean,sentiment_label,topic_general_label
0,This doesn't surprise me since the entire hist...,Negative,Fast Fashion Shopping Habits
1,Okay then why work for these factories.....or ...,Negative,Fast Fashion Shopping Habits
2,Please do not be this ignorant. The US and oth...,Negative,Fast Fashion Shopping Habits
3,"@sbradshaw1886 Still a choice ...Ignorence, re...",Neutral,Environmental Impact
4,They fuel buying addiction. Same with Temu. No...,Negative,Topic 11


## Summary

**What we did in this step:**
- Ran BERTopic on all 30,609 comments → discovered general themes in fast fashion discussions
- Ran BERTopic separately on Positive, Negative, and Neutral comment groups
- Manually interpreted and labelled each discovered topic with a meaningful name
- Saved results as `youtube_comments_topics.csv`

**New columns added:**
- `topic_general` — topic ID from full dataset modeling
- `topic_general_label` — human-readable topic name
- `topic_positive` — topic ID from positive comments only
- `topic_negative` — topic ID from negative comments only
- `topic_neutral` — topic ID from neutral comments only

**Key insight this enables for your dashboard:**
- Show a topic breakdown chart for all comments
- Let users filter by sentiment and see how topics shift
- Power the "Topic Explorer" tab in Streamlit

**What to say in your report:**
*"BERTopic was applied to the full English comment corpus (n=30,609) to identify latent themes.
The model was additionally run per sentiment group to reveal whether positive and negative
commenters discuss distinct aspects of fast fashion. Topics were manually interpreted and
labelled to ensure meaningful representation."*

**Next Step → Step 5: RAG Framework**
